# SDC2DSquare Position-Addressed Thermodynamics

This notebook mirrors the system built in `rgrow/examples/sdc2d_position_gui.py`, but uses the exact finite-grid thermodynamics API instead of launching the GUI.

The model is an 8x8 scaffold. Each scaffold position has position-specific `0` and `1` tile types, except the bottom-left site, which is fixed to a `0` tile by default. Exact 2D thermodynamics are exponential in the smaller grid dimension, so these examples are useful for small systems and parameter studies, not arbitrary large grids.

In [ ]:
from __future__ import annotations

from dataclasses import dataclass

import matplotlib.pyplot as plt
import numpy as np

from rgrow.sdc2d import SDC2DSquare, SDC2DParams, SDC2DStrand

In [ ]:
N = 8
CONC = 1e-6
BOTTOM_LEFT = (N - 1, 0)
R_KCAL_PER_MOL_K = 1.98720425864083e-3


@dataclass(frozen=True)
class GlueSet:
    west: str | None
    north: str | None
    east: str | None
    south: str | None


def position_glue(row: int, col: int) -> str:
    return f"pos_{row}_{col}"


def lateral_glues(row: int, col: int, bit: int, *, n: int = N) -> GlueSet:
    west = f"h_{row}_{col - 1}_{bit}*" if col > 0 else None
    east = f"h_{row}_{col}_{bit}" if col < n - 1 else None
    north = f"v_{row - 1}_{col}_{bit}*" if row > 0 else None
    south = f"v_{row}_{col}_{bit}" if row < n - 1 else None
    return GlueSet(west=west, north=north, east=east, south=south)


def make_strand(row: int, col: int, bit: int, *, name: str | None = None, n: int = N) -> SDC2DStrand:
    glues = lateral_glues(row, col, bit, n=n)
    color = "#2f6fdd" if bit == 0 else "#d69225"
    return SDC2DStrand(
        name=name or f"t_{row}_{col}_{bit}",
        color=color,
        concentration=CONC,
        west_glue=glues.west,
        north_glue=glues.north,
        east_glue=glues.east,
        south_glue=glues.south,
        bottom_glue=position_glue(row, col),
    )


def build_system(*, gc: float = -7.0, gs: float = -8.0, gi: float = -20.0, seed_mode: bool = True, temperature: float = 37.0, n: int = N):
    bottom_left = (n - 1, 0)
    strands: list[SDC2DStrand] = []
    seed: list[tuple[int, int, str]] = []
    glue_dg37_ds: dict[str, tuple[float, float]] = {}

    for row in range(n):
        for col in range(n):
            is_bottom_left = (row, col) == bottom_left
            if is_bottom_left and seed_mode:
                seed_name = f"seed_{row}_{col}_0"
                strands.append(make_strand(row, col, 0, name=seed_name, n=n))
                seed.append((row, col, seed_name))
            elif is_bottom_left:
                strands.append(make_strand(row, col, 0, n=n))
            else:
                strands.append(make_strand(row, col, 0, n=n))
                strands.append(make_strand(row, col, 1, n=n))

            scaffold_energy = gs if seed_mode or not is_bottom_left else gi
            glue_dg37_ds[position_glue(row, col)] = (scaffold_energy, 0.0)

    for row in range(n):
        for col in range(n - 1):
            for bit in (0, 1):
                glue_dg37_ds[f"h_{row}_{col}_{bit}"] = (gc, 0.0)
    for row in range(n - 1):
        for col in range(n):
            for bit in (0, 1):
                glue_dg37_ds[f"v_{row}_{col}_{bit}"] = (gc, 0.0)

    scaffold = [[f"{position_glue(row, col)}*" for col in range(n)] for row in range(n)]
    params = SDC2DParams(
        strands=strands,
        scaffold=scaffold,
        scaffold_concentration=1e-100,
        glue_dg37_ds=glue_dg37_ds,
        k_f=1e6,
        temperature=temperature,
        seed=seed,
    )
    sys = SDC2DSquare(params)
    tile_by_name = {name: idx for idx, name in enumerate(sys.strand_names)}
    tile_by_pos_bit: dict[tuple[int, int, int], int] = {}
    bit_by_tile: dict[int, int] = {0: -1}
    for row in range(n):
        for col in range(n):
            bits = (0,) if (row, col) == bottom_left else (0, 1)
            for bit in bits:
                if (row, col) == bottom_left and seed_mode:
                    name = f"seed_{row}_{col}_0"
                else:
                    name = f"t_{row}_{col}_{bit}"
                tile = tile_by_name[name]
                tile_by_pos_bit[(row, col, bit)] = tile
                bit_by_tile[tile] = bit
    return sys, {"n": n, "bottom_left": bottom_left, "tile_by_pos_bit": tile_by_pos_bit, "bit_by_tile": bit_by_tile, "temperature": temperature}


def empty_constraints(n: int = N) -> list[list[list[int]]]:
    return [[[] for _ in range(n)] for _ in range(n)]


def state_from_pattern(info, pattern) -> list[list[int]]:
    n = info["n"]
    bottom_left = info["bottom_left"]
    tile_by_pos_bit = info["tile_by_pos_bit"]
    state = [[0 for _ in range(n)] for _ in range(n)]
    for row in range(n):
        for col in range(n):
            bit = 0 if (row, col) == bottom_left else int(pattern(row, col))
            state[row][col] = tile_by_pos_bit[(row, col, bit)]
    return state


def bits_from_state(state: list[list[int]], info) -> np.ndarray:
    bit_by_tile = info["bit_by_tile"]
    return np.array([[bit_by_tile.get(tile, np.nan) for tile in row] for row in state], dtype=float)


def plot_bits(bits: np.ndarray, title: str):
    fig, ax = plt.subplots(figsize=(4.8, 4.8))
    cmap = plt.get_cmap("coolwarm", 3)
    image = ax.imshow(bits, vmin=-1, vmax=1, cmap=cmap)
    ax.set_xticks(range(bits.shape[1]))
    ax.set_yticks(range(bits.shape[0]))
    ax.set_title(title)
    cbar = fig.colorbar(image, ax=ax, ticks=[-1, 0, 1], fraction=0.046, pad=0.04)
    cbar.ax.set_yticklabels(["empty", "0", "1"])
    return fig, ax

## Build the default seeded system

The defaults match the GUI example: `gc=-7`, `gs=-8`, `gi=-20`, `temperature=37`, and seed mode enabled.

In [ ]:
sys, info = build_system()
print(f"grid: {sys.nrows()}x{sys.ncols()}")
print(f"tile types, including empty tile 0: {sys.n_strands()}")
print(f"bottom-left allowed friends: {sys.friends_at(N - 1, 0)}")
print(f"center allowed friends: {sys.friends_at(3, 3)}")

## Minimum free energy configuration

`mfe_config()` returns a 2D tile-id state and its free energy in kcal/mol. Here the seed selects the `0` phase, and like-valued lateral bonds make a filled `0` region favorable.

In [ ]:
mfe_state, mfe_g = sys.mfe_config()
mfe_bits = bits_from_state(mfe_state, info)
print(f"MFE free energy: {mfe_g:.3f} kcal/mol")
print(f"state_g(mfe_state): {sys.state_g(mfe_state):.3f} kcal/mol")
plot_bits(mfe_bits, "MFE bit pattern");

## Partition function and ensemble free energy

The partition function sums over all legal configurations exactly using the frontier dynamic program. For larger systems, prefer `log_partition_function()` because `partition_function()` can overflow.

In [ ]:
log_z = sys.log_partition_function()
z = sys.partition_function()
rt = R_KCAL_PER_MOL_K * (info["temperature"] + 273.15)
ensemble_g = -rt * log_z

print(f"log Z: {log_z:.3f}")
print(f"Z: {z:.3e}")
print(f"ensemble free energy -RT log Z: {ensemble_g:.3f} kcal/mol")
print(f"MFE gap above ensemble free energy: {mfe_g - ensemble_g:.3f} kcal/mol")

## Probabilities of named configurations

The exact probability of a concrete state is `exp(-G/RT) / Z`. The all-zero state is compatible with the seed; the all-one state keeps the seeded bottom-left tile at `0` and places `1` everywhere else.

In [ ]:
all_zero = state_from_pattern(info, lambda row, col: 0)
all_one_except_seed = state_from_pattern(info, lambda row, col: 1)
checkerboard = state_from_pattern(info, lambda row, col: (row + col) % 2)

rows = []
for label, state in [
    ("all zero", all_zero),
    ("all one except seed", all_one_except_seed),
    ("checkerboard", checkerboard),
    ("MFE", mfe_state),
]:
    rows.append((label, sys.state_g(state), sys.probability_of_state(state)))

for label, g, p in rows:
    print(f"{label:20s} G={g:10.3f} kcal/mol   P={p:.3e}")

## Constrained partition functions

A constraint is a `list[list[list[int]]]`. An empty innermost list means unconstrained; otherwise it lists the allowed tile IDs at that position. This makes it possible to ask for probabilities of events such as a site being empty, a site being `1`, or a block being fixed to `0`.

In [ ]:
def probability_site_is(sys: SDC2DSquare, info, row: int, col: int, allowed_tiles: list[int]) -> float:
    constraints = empty_constraints(info["n"])
    constraints[row][col] = allowed_tiles
    return sys.probability_of_constrained_configurations(constraints)


center = (N // 2, N // 2)
center_zero_tile = info["tile_by_pos_bit"][(*center, 0)]
center_one_tile = info["tile_by_pos_bit"][(*center, 1)]

print(f"P(center empty): {probability_site_is(sys, info, *center, [0]):.6f}")
print(f"P(center bit 0): {probability_site_is(sys, info, *center, [center_zero_tile]):.6f}")
print(f"P(center bit 1): {probability_site_is(sys, info, *center, [center_one_tile]):.6f}")

block_constraints = empty_constraints(N)
for row in range(2, 6):
    for col in range(2, 6):
        block_constraints[row][col] = [info["tile_by_pos_bit"][(row, col, 0)]]
print(f"P(central 4x4 block fixed to bit 0): {sys.probability_of_constrained_configurations(block_constraints):.6f}")

## Site marginals as a heat map

Each marginal below is computed by one constrained partition function. This is exact, but it repeats the DP once per queried event.

In [ ]:
p_bit1 = np.zeros((N, N), dtype=float)
p_occupied = np.zeros((N, N), dtype=float)

for row in range(N):
    for col in range(N):
        p_empty = probability_site_is(sys, info, row, col, [0])
        p_occupied[row, col] = 1.0 - p_empty
        if (row, col, 1) in info["tile_by_pos_bit"]:
            one_tile = info["tile_by_pos_bit"][(row, col, 1)]
            p_bit1[row, col] = probability_site_is(sys, info, row, col, [one_tile])

fig, axes = plt.subplots(1, 2, figsize=(10, 4), constrained_layout=True)
for ax, data, title in [
    (axes[0], p_occupied, "P(occupied)"),
    (axes[1], p_bit1, "P(bit 1)"),
]:
    im = ax.imshow(data, vmin=0, vmax=1, cmap="viridis")
    ax.set_xticks(range(N))
    ax.set_yticks(range(N))
    ax.set_title(title)
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

## Seeded system versus strong unpinned anchor

The GUI script can either pin the bottom-left tile as a seed, or leave it unpinned with a stronger scaffold bond `gi`. These ensembles are not identical: the seed removes the empty bottom-left state, while the anchor mode merely makes it unlikely.

In [ ]:
seeded_sys, seeded_info = build_system(seed_mode=True)
anchor_sys, anchor_info = build_system(seed_mode=False)

for label, model, model_info in [
    ("seeded", seeded_sys, seeded_info),
    ("strong unpinned anchor", anchor_sys, anchor_info),
]:
    state, g = model.mfe_config()
    p_anchor_empty = probability_site_is(model, model_info, N - 1, 0, [0])
    print(f"{label:22s} logZ={model.log_partition_function():9.3f}   MFE={g:9.3f} kcal/mol   P(bottom-left empty)={p_anchor_empty:.3e}")

## Lateral bond strength sweep

Changing `gc` tunes the cost of domain walls. Stronger lateral bonds make the seed-selected phase dominate more sharply.

In [ ]:
gc_values = np.linspace(-2.0, -9.0, 8)
sweep = []
for gc in gc_values:
    model, model_info = build_system(gc=float(gc))
    state, mfe = model.mfe_config()
    log_z = model.log_partition_function()
    center_one = model_info["tile_by_pos_bit"][(*center, 1)]
    p_center_one = probability_site_is(model, model_info, *center, [center_one])
    sweep.append((gc, mfe, -rt * log_z, p_center_one))

sweep = np.array(sweep)
fig, axes = plt.subplots(1, 2, figsize=(10, 4), constrained_layout=True)
axes[0].plot(sweep[:, 0], sweep[:, 1], marker="o", label="MFE")
axes[0].plot(sweep[:, 0], sweep[:, 2], marker="o", label="-RT log Z")
axes[0].invert_xaxis()
axes[0].set_xlabel("gc (kcal/mol)")
axes[0].set_ylabel("free energy (kcal/mol)")
axes[0].legend()

axes[1].plot(sweep[:, 0], sweep[:, 3], marker="o")
axes[1].invert_xaxis()
axes[1].set_xlabel("gc (kcal/mol)")
axes[1].set_ylabel("P(center bit 1)")
axes[1].set_ylim(-0.02, 1.02)